# Sparse Computation and Graphs

Work with sparse data structures on the GPU: CSR, COO, and ELL matrix formats, sparse matrix-vector multiply (SpMV), and parallel graph algorithms like BFS.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/cuda-sparse-graphs)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Then run the install cell below.

In [ ]:
!pip install numba

---

## Lesson examples (optional)

These are the code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.

### Lesson example: Sparse Matrix Formats

In [ ]:
!pip install numba
import numpy as np
from numba import cuda
import numba
from scipy import sparse

# ============================================================
# Create and explore sparse matrix formats
# ============================================================
print("=" * 60)
print("Sparse Matrix Formats")
print("=" * 60)

# Create a sparse matrix
np.random.seed(42)
N = 1000
density = 0.01
A_coo = sparse.random(N, N, density=density, format='coo', dtype=np.float32)
print(f"Matrix: {A_coo.shape}, nnz={A_coo.nnz}, density={density}")

# Convert to CSR
A_csr = A_coo.tocsr()
print(f"\nCSR format:")
print(f"  rowptr: shape={A_csr.indptr.shape}, dtype={A_csr.indptr.dtype}")
print(f"  col:    shape={A_csr.indices.shape}, dtype={A_csr.indices.dtype}")
print(f"  val:    shape={A_csr.data.shape}, dtype={A_csr.data.dtype}")
print(f"  Row 0 has {A_csr.indptr[1] - A_csr.indptr[0]} non-zeros")
print(f"  Row 5 has {A_csr.indptr[6] - A_csr.indptr[5]} non-zeros")

# Show row length distribution
row_lengths = np.diff(A_csr.indptr)
print(f"\nRow length statistics:")
print(f"  Min: {row_lengths.min()}, Max: {row_lengths.max()}, "
      f"Mean: {row_lengths.mean():.1f}, Std: {row_lengths.std():.1f}")

# COO format
print(f"\nCOO format:")
print(f"  row: shape={A_coo.row.shape}")
print(f"  col: shape={A_coo.col.shape}")
print(f"  val: shape={A_coo.data.shape}")
print(f"  First 5 entries: ({A_coo.row[:5]}, {A_coo.col[:5]}) = {A_coo.data[:5]}")

# ============================================================
# ELL format construction
# ============================================================
print("\n" + "=" * 60)
print("ELL Format Construction")
print("=" * 60)

def csr_to_ell(csr_matrix):
    """Convert CSR to ELL format (column-major for GPU coalescing)."""
    num_rows = csr_matrix.shape[0]
    row_lengths = np.diff(csr_matrix.indptr)
    max_nnz = int(row_lengths.max())
    
    # Column-major storage: ell_col[row + col_in_row * num_rows]
    ell_col = np.full(num_rows * max_nnz, -1, dtype=np.int32)
    ell_val = np.zeros(num_rows * max_nnz, dtype=np.float32)
    
    for i in range(num_rows):
        start = csr_matrix.indptr[i]
        end = csr_matrix.indptr[i + 1]
        length = end - start
        for j in range(length):
            # Column-major index: row i, column-slot j
            idx = i + j * num_rows
            ell_col[idx] = csr_matrix.indices[start + j]
            ell_val[idx] = csr_matrix.data[start + j]
    
    return ell_col, ell_val, max_nnz

ell_col, ell_val, max_nnz = csr_to_ell(A_csr)
print(f"ELL: max_nnz_per_row={max_nnz}, total storage={len(ell_col)} entries")

# ============================================================
# Memory comparison
# ============================================================
print("\n" + "=" * 60)
print("Memory Comparison")
print("=" * 60)

nnz = A_coo.nnz
dense_mem = N * N * 4  # float32
coo_mem = 3 * nnz * 4  # row + col + val (int32 + int32 + float32)
csr_mem = (N + 1) * 4 + 2 * nnz * 4  # rowptr + col + val
ell_mem = 2 * N * max_nnz * 4  # col + val (padded)

print(f"Dense:  {dense_mem:>12,} bytes ({dense_mem/1e6:.1f} MB)")
print(f"COO:    {coo_mem:>12,} bytes ({coo_mem/1e6:.1f} MB)  "
      f"({dense_mem/coo_mem:.0f}x smaller)")
print(f"CSR:    {csr_mem:>12,} bytes ({csr_mem/1e6:.1f} MB)  "
      f"({dense_mem/csr_mem:.0f}x smaller)")
print(f"ELL:    {ell_mem:>12,} bytes ({ell_mem/1e6:.1f} MB)  "
      f"({dense_mem/ell_mem:.0f}x smaller)")

# ============================================================
# Show ELL padding waste for power-law distribution
# ============================================================
print("\n" + "=" * 60)
print("ELL Padding Waste (Power-Law Distribution)")
print("=" * 60)

# Create matrix with power-law row lengths (like social network)
from scipy.stats import pareto
np.random.seed(42)
row_lens = np.clip(pareto.rvs(2.5, size=1000) * 5, 1, 500).astype(int)
total_nnz = int(row_lens.sum())
max_len = int(row_lens.max())

csr_mem_pl = (1001 + 2 * total_nnz) * 4
ell_mem_pl = 2 * 1000 * max_len * 4

print(f"Row lengths: min={row_lens.min()}, max={max_len}, mean={row_lens.mean():.1f}")
print(f"Total nnz: {total_nnz}")
print(f"CSR memory: {csr_mem_pl/1e6:.1f} MB")
print(f"ELL memory: {ell_mem_pl/1e6:.1f} MB ({ell_mem_pl/csr_mem_pl:.1f}x CSR)")
print(f"ELL padding waste: {(1 - total_nnz/(1000*max_len))*100:.1f}%")
print("ELL is wasteful for power-law distributions -- use HYB or CSR instead.")

### Lesson example: SpMV: Sparse Matrix-Vector Multiply

In [ ]:
!pip install numba
import numpy as np
from numba import cuda
import numba
from scipy import sparse
import time

# ============================================================
# CSR SpMV Kernel: One thread per row
# ============================================================
@cuda.jit
def spmv_csr_kernel(rowptr, col, val, x, y, num_rows):
    """Sparse matrix-vector multiply using CSR format.
    One thread per row.
    """
    row = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if row < num_rows:
        total = 0.0
        for k in range(rowptr[row], rowptr[row + 1]):
            total += val[k] * x[col[k]]
        y[row] = total


# ============================================================
# ELL SpMV Kernel
# ============================================================
@cuda.jit
def spmv_ell_kernel(ell_col, ell_val, x, y, num_rows, max_nnz):
    """Sparse matrix-vector multiply using ELL format.
    Column-major storage for coalescing.
    """
    row = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if row < num_rows:
        total = 0.0
        for j in range(max_nnz):
            idx = row + j * num_rows  # Column-major
            c = ell_col[idx]
            if c >= 0:
                total += ell_val[idx] * x[c]
        y[row] = total


# ============================================================
# Helper: CSR to ELL conversion
# ============================================================
def csr_to_ell(csr_matrix):
    num_rows = csr_matrix.shape[0]
    row_lengths = np.diff(csr_matrix.indptr)
    max_nnz = int(row_lengths.max())
    
    ell_col = np.full(num_rows * max_nnz, -1, dtype=np.int32)
    ell_val = np.zeros(num_rows * max_nnz, dtype=np.float32)
    
    for i in range(num_rows):
        start = csr_matrix.indptr[i]
        end = csr_matrix.indptr[i + 1]
        for j in range(end - start):
            idx = i + j * num_rows
            ell_col[idx] = csr_matrix.indices[start + j]
            ell_val[idx] = csr_matrix.data[start + j]
    
    return ell_col, ell_val, max_nnz


# ============================================================
# Create test matrix and verify
# ============================================================
print("=" * 60)
print("SpMV: Sparse Matrix-Vector Multiply")
print("=" * 60)

np.random.seed(42)
N = 5000
density = 0.01
A = sparse.random(N, N, density=density, format='csr', dtype=np.float32)
x = np.random.randn(N).astype(np.float32)

print(f"Matrix: {N}x{N}, nnz={A.nnz}, density={density}")
print(f"Row lengths: min={np.diff(A.indptr).min()}, max={np.diff(A.indptr).max()}, "
      f"mean={np.diff(A.indptr).mean():.1f}")

# Reference result
y_ref = A.dot(x)

# ---- CSR SpMV ----
d_rowptr = cuda.to_device(A.indptr.astype(np.int32))
d_col = cuda.to_device(A.indices.astype(np.int32))
d_val = cuda.to_device(A.data.astype(np.float32))
d_x = cuda.to_device(x)
d_y = cuda.device_array(N, dtype=np.float32)

threads = 256
blocks = (N + threads - 1) // threads
spmv_csr_kernel[blocks, threads](d_rowptr, d_col, d_val, d_x, d_y, N)
y_csr = d_y.copy_to_host()
print(f"\nCSR SpMV: max error = {np.max(np.abs(y_csr - y_ref)):.6e}, "
      f"correct = {np.allclose(y_csr, y_ref, rtol=1e-4)}")

# ---- ELL SpMV ----
ell_col, ell_val, max_nnz = csr_to_ell(A)
d_ell_col = cuda.to_device(ell_col)
d_ell_val = cuda.to_device(ell_val)
d_y2 = cuda.device_array(N, dtype=np.float32)

spmv_ell_kernel[blocks, threads](d_ell_col, d_ell_val, d_x, d_y2, N, max_nnz)
y_ell = d_y2.copy_to_host()
print(f"ELL SpMV: max error = {np.max(np.abs(y_ell - y_ref)):.6e}, "
      f"correct = {np.allclose(y_ell, y_ref, rtol=1e-4)}")

# ============================================================
# Benchmark
# ============================================================
print("\n" + "=" * 60)
print("Performance Benchmark")
print("=" * 60)

iters = 200

# Warm up
spmv_csr_kernel[blocks, threads](d_rowptr, d_col, d_val, d_x, d_y, N)
spmv_ell_kernel[blocks, threads](d_ell_col, d_ell_val, d_x, d_y2, N, max_nnz)
cuda.synchronize()

# CSR timing
start = time.perf_counter()
for _ in range(iters):
    spmv_csr_kernel[blocks, threads](d_rowptr, d_col, d_val, d_x, d_y, N)
cuda.synchronize()
csr_time = (time.perf_counter() - start) / iters

# ELL timing
start = time.perf_counter()
for _ in range(iters):
    spmv_ell_kernel[blocks, threads](d_ell_col, d_ell_val, d_x, d_y2, N, max_nnz)
cuda.synchronize()
ell_time = (time.perf_counter() - start) / iters

# Dense matmul timing (for comparison)
A_dense = cuda.to_device(A.toarray().astype(np.float32))
d_y3 = cuda.device_array(N, dtype=np.float32)

@cuda.jit
def dense_matvec(A, x, y, N):
    row = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if row < N:
        total = 0.0
        for j in range(N):
            total += A[row, j] * x[j]
        y[row] = total

dense_matvec[blocks, threads](A_dense, d_x, d_y3, N)
cuda.synchronize()
start = time.perf_counter()
for _ in range(iters):
    dense_matvec[blocks, threads](A_dense, d_x, d_y3, N)
cuda.synchronize()
dense_time = (time.perf_counter() - start) / iters

flops = 2 * A.nnz  # multiply + add per non-zero
flops_dense = 2 * N * N

print(f"CSR SpMV:    {csr_time*1e6:.1f} us  ({flops/csr_time/1e9:.2f} GFLOP/s)")
print(f"ELL SpMV:    {ell_time*1e6:.1f} us  ({flops/ell_time/1e9:.2f} GFLOP/s)")
print(f"Dense MV:    {dense_time*1e6:.1f} us  ({flops_dense/dense_time/1e9:.2f} GFLOP/s)")
print(f"\nSpeedup: CSR is {dense_time/csr_time:.1f}x faster than dense")
print(f"         ELL is {dense_time/ell_time:.1f}x faster than dense")
print(f"\nSparse saves {(1 - density)*100:.1f}% of compute vs dense!")

### Lesson example: Graph Algorithms on GPU

In [ ]:
!pip install numba
import numpy as np
from numba import cuda
import numba
from scipy import sparse
import time

# ============================================================
# Build a graph (CSR format)
# ============================================================
def build_random_graph(num_nodes, avg_degree, seed=42):
    """Build a random graph in CSR format."""
    np.random.seed(seed)
    # Create random edges
    rows = []
    cols = []
    for i in range(num_nodes):
        # Each node gets ~avg_degree random neighbors
        num_neighbors = np.random.poisson(avg_degree)
        neighbors = np.random.randint(0, num_nodes, size=num_neighbors)
        for j in neighbors:
            if j != i:  # No self-loops
                rows.append(i)
                cols.append(j)
    
    # Build CSR
    A = sparse.csr_matrix(
        (np.ones(len(rows), dtype=np.float32), (rows, cols)),
        shape=(num_nodes, num_nodes)
    )
    return A

# ============================================================
# Level-synchronous BFS on GPU
# ============================================================
@cuda.jit
def bfs_level_kernel(rowptr, col, distance, num_nodes, level, changed):
    """Explore all nodes at 'level', discover neighbors at 'level+1'."""
    tid = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if tid < num_nodes and distance[tid] == level:
        for k in range(rowptr[tid], rowptr[tid + 1]):
            neighbor = col[k]
            if distance[neighbor] == -1:
                distance[neighbor] = level + 1
                changed[0] = 1


def gpu_bfs(rowptr, col, source, num_nodes):
    """Level-synchronous BFS on GPU."""
    d_rowptr = cuda.to_device(rowptr.astype(np.int32))
    d_col = cuda.to_device(col.astype(np.int32))
    
    distance = np.full(num_nodes, -1, dtype=np.int32)
    distance[source] = 0
    d_distance = cuda.to_device(distance)
    
    threads = 256
    blocks = (num_nodes + threads - 1) // threads
    
    level = 0
    while True:
        d_changed = cuda.to_device(np.array([0], dtype=np.int32))
        bfs_level_kernel[blocks, threads](
            d_rowptr, d_col, d_distance, num_nodes, level, d_changed
        )
        cuda.synchronize()
        
        changed = d_changed.copy_to_host()[0]
        if changed == 0:
            break
        level += 1
    
    return d_distance.copy_to_host(), level


# ============================================================
# CPU BFS for reference
# ============================================================
def cpu_bfs(rowptr, col, source, num_nodes):
    from collections import deque
    distance = np.full(num_nodes, -1, dtype=np.int32)
    distance[source] = 0
    queue = deque([source])
    max_level = 0
    
    while queue:
        node = queue.popleft()
        for k in range(rowptr[node], rowptr[node + 1]):
            neighbor = col[k]
            if distance[neighbor] == -1:
                distance[neighbor] = distance[node] + 1
                max_level = max(max_level, distance[neighbor])
                queue.append(neighbor)
    
    return distance, max_level


# ============================================================
# Test and benchmark
# ============================================================
print("=" * 60)
print("Graph BFS: GPU vs CPU")
print("=" * 60)

N = 10000
avg_deg = 10
A = build_random_graph(N, avg_deg)
rowptr = A.indptr
col = A.indices
print(f"Graph: {N} nodes, {A.nnz} edges, avg degree={A.nnz/N:.1f}")

# Check row (node) degree distribution
degrees = np.diff(rowptr)
print(f"Degree distribution: min={degrees.min()}, max={degrees.max()}, "
      f"mean={degrees.mean():.1f}, std={degrees.std():.1f}")

source = 0

# CPU BFS
cpu_dist, cpu_levels = cpu_bfs(rowptr, col, source, N)
reachable = np.sum(cpu_dist >= 0)
print(f"\nBFS from node {source}: {reachable} reachable nodes, {cpu_levels} levels")

# GPU BFS
gpu_dist, gpu_levels = gpu_bfs(rowptr, col, source, N)

# Verify
match = np.array_equal(cpu_dist, gpu_dist)
print(f"GPU matches CPU: {match}")
if not match:
    diff_count = np.sum(cpu_dist != gpu_dist)
    print(f"  Differences: {diff_count} nodes")

# Level statistics
print(f"\nNodes per BFS level:")
for level in range(cpu_levels + 1):
    count = np.sum(cpu_dist == level)
    bar = '#' * min(count // max(N // 100, 1), 50)
    print(f"  Level {level:>3}: {count:>6} nodes {bar}")

# ============================================================
# Benchmark
# ============================================================
print("\n" + "=" * 60)
print("Benchmark")
print("=" * 60)

iters = 20

# CPU timing
start = time.perf_counter()
for _ in range(iters):
    cpu_bfs(rowptr, col, source, N)
cpu_time = (time.perf_counter() - start) / iters

# GPU timing  
_ = gpu_bfs(rowptr, col, source, N)  # warm up
cuda.synchronize()
start = time.perf_counter()
for _ in range(iters):
    gpu_bfs(rowptr, col, source, N)
cuda.synchronize()
gpu_time = (time.perf_counter() - start) / iters

print(f"CPU BFS: {cpu_time*1e3:.2f} ms")
print(f"GPU BFS: {gpu_time*1e3:.2f} ms")
print(f"Speedup: {cpu_time/gpu_time:.2f}x")
print(f"\nNote: Level-synchronous BFS has kernel launch overhead per level.")
print(f"For larger graphs (100K+ nodes), GPU speedup increases significantly.")
print(f"Production libraries (cuGraph) achieve 5-20x speedup.")

---

## Exercise: CSR SpMV and Dense Matmul Benchmark


In [ ]:
import numpy as np
from numba import cuda
import numba
from scipy import sparse
import time

# ============================================================
# TODO 1: CSR SpMV Kernel
# ============================================================
@cuda.jit
def spmv_csr_kernel(rowptr, col, val, x, y, num_rows):
    """Sparse matrix-vector multiply: y = A @ x (CSR format).
    One thread per row.
    """
    row = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    # TODO: For each row, iterate over its non-zeros and compute dot product
    pass


# ============================================================
# TODO 2: Dense Matrix-Vector Multiply Kernel
# ============================================================
@cuda.jit
def dense_matvec_kernel(A, x, y, N):
    """Dense matrix-vector multiply: y = A @ x.
    One thread per row.
    """
    row = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    # TODO: Compute dot product of row with x
    pass


# ============================================================
# Testing
# ============================================================
np.random.seed(42)
N = 5000
density = 0.01

# Create sparse matrix and vector
A_sparse = sparse.random(N, N, density=density, format='csr', dtype=np.float32)
x = np.random.randn(N).astype(np.float32)

print("Testing correctness...")
# TODO: Test CSR SpMV against scipy A_sparse.dot(x)
# TODO: Test dense MV against A_sparse.toarray() @ x

# ============================================================
# Benchmarking
# ============================================================
print("\nBenchmarking across densities...")
for density in [0.001, 0.01, 0.05, 0.1]:
    # TODO: Create matrix, benchmark both kernels
    # TODO: Report time, GFLOP/s, and sparse vs dense speedup
    pass

## Your tasks

Implement CSR-based sparse matrix-vector multiply on the GPU and benchmark it against dense matrix-vector multiply.

**Requirements:**
1. Implement a CSR SpMV kernel (one thread per row)
2. Implement a simple dense matrix-vector multiply kernel for comparison
3. Create sparse test matrices of varying sizes and densities
4. Verify both kernels produce correct results (compare against scipy)
5. Benchmark SpMV vs dense MV and compute:
   - Execution time for both
   - GFLOP/s for both
   - Speedup of sparse over dense
   - Effective memory bandwidth for SpMV
6. Show how the density threshold affects the sparse vs dense tradeoff

**Hints:**
- CSR SpMV: `y[row] = sum(val[k] * x[col[k]] for k in range(rowptr[row], rowptr[row+1]))`
- Dense MV: `y[row] = sum(A[row, j] * x[j] for j in range(N))`
- Use scipy.sparse.random() to create test matrices
- SpMV FLOPs = 2 * nnz; Dense FLOPs = 2 * N * N
- SpMV bytes loaded per non-zero: 12 (val + col + x element)
- Test densities: 0.001, 0.01, 0.05, 0.1, 0.5

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/cuda-sparse-graphs) for the solution and discussion.